# (g,s)-Dependent LOLR Test: $\sigma = 4\%$, $\Delta(g_L,s_B)=30\%$, $\Delta=0\%$ elsewhere

In [1]:
using Statistics
using Serialization
using Plots
include("src/simple_s_lolr_v2.jl")
Threads.nthreads()

8

## Scenario

In [2]:
# Scenario test notebook.
# LOLR spread is σ = 4% in all states.
# Δ is large only in (no default, g_L, s_B): 30%.
# Δ = 0% in all other (d, g, s) states.
Δ_dgs = fill(0.00, 2, 2, 2)
Δ_dgs[1, 1, 1] = 0.30
model = init_model(Model(
    Nb = 800,
    Nl = 20,
    Ne = 5,
    R_l = 1.075,
    Δ_dgs = Δ_dgs,
))
model.max_iter = 80
model.max_iter_vd = 100
model.max_iter_x = 100
model.pub = 0.65
model


Model(800, 20, 2, 2, 5, -0.05, 1.0, 0.0, 0.3027692307692307, [-0.05, -0.048685857321652065, -0.04737171464330413, -0.0460575719649562, -0.04474342928660826, -0.04342928660826033, -0.04211514392991239, -0.04080100125156445, -0.03948685857321652, -0.038172715894868585  …  0.9881727158948685, 0.9894868585732165, 0.9908010012515645, 0.9921151439299124, 0.9934292866082604, 0.9947434292866083, 0.9960575719649561, 0.9973717146433041, 0.998685857321652, 1.0], [0.0, 0.015935222672064774, 0.03187044534412955, 0.04780566801619432, 0.0637408906882591, 0.07967611336032386, 0.09561133603238864, 0.1115465587044534, 0.1274817813765182, 0.14341700404858296, 0.15935222672064772, 0.17528744939271249, 0.19122267206477728, 0.20715789473684204, 0.2230931174089068, 0.2390283400809716, 0.2549635627530364, 0.27089878542510115, 0.2868340080971659, 0.3027692307692307], [0.96, 1.04], [-2.5, -1.25, 0.0, 1.25, 2.5], [0.6 0.4; 0.25 0.75], [0.25 0.75; 0.25 0.75], [0.030396361765261396, 0.2355891672834392, 0.468028941

## Solve Model

In [3]:
sol = solve_model(model; verbose = true)
println("Mean default probability in the state grid = ", mean(sol.d))
println("Final outer error = ", sol.outer_errs[end])

iter=1, vnd_err=34.28699119213285, vd_err=6.586529810448383e-7, x_err=8.912555766737995e-7, damp=0.9
iter=2, vnd_err=2594.8467817401474, vd_err=8.501175638997438e-7, x_err=0.00015225252810725787, damp=0.9
iter=3, vnd_err=13011.38750613029, vd_err=7.578296603583112e-7, x_err=1.3971139291621415e-5, damp=0.9
iter=4, vnd_err=11538.432795275361, vd_err=9.235054125866782e-7, x_err=5.872815684593302e-6, damp=0.9
iter=5, vnd_err=15103.385633256215, vd_err=7.089379820968134e-7, x_err=7.67345332408631e-6, damp=0.9
iter=6, vnd_err=38205.96096691235, vd_err=8.927951196113781e-7, x_err=7.186989118002085e-6, damp=0.9
iter=7, vnd_err=34012.45128583595, vd_err=7.584316161768356e-7, x_err=6.644979733433365e-6, damp=0.9
iter=8, vnd_err=26601.66319061135, vd_err=8.892961727013926e-7, x_err=2.1042613364830043e-6, damp=0.9
iter=9, vnd_err=10377.864412468169, vd_err=6.813398316651842e-7, x_err=4.6226737121790595e-6, damp=0.9
iter=10, vnd_err=4670.388420175034, vd_err=6.97418526485194e-7, x_err=1.57104150477

## Low-Growth Policy Plots

In [4]:

# Low-growth plots for the current scenario.
gi_low = argmin(model.g)
ei_mid = cld(model.Ne, 2)
s_bad = 1
s_good = model.Ns
li_zero = argmin(abs.(model.l))
lprime_zero = li_zero

mask_bad = sol.schedule_mask[:, lprime_zero, gi_low, s_bad]
mask_good = sol.schedule_mask[:, lprime_zero, gi_low, s_good]
schedule_bad = (n = sol.n[mask_bad, lprime_zero, gi_low, s_bad], R = sol.R[mask_bad, lprime_zero, gi_low, s_bad])
schedule_good = (n = sol.n[mask_good, lprime_zero, gi_low, s_good], R = sol.R[mask_good, lprime_zero, gi_low, s_good])

p_schedule = plot(
    schedule_bad.n,
    schedule_bad.R .- 1,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "private issuance n",
    ylabel = "private rate R - 1",
    title = "Private schedule at l' = 0 (σ = 4%, Δ(g_L,s_B)=30%, Δ=0 elsewhere) ",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (1100, 650),
)
plot!(p_schedule, schedule_good.n, schedule_good.R .- 1, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

bad_bp = sol.b_policy_idx[:, li_zero, gi_low, s_bad, ei_mid]
bad_lp = sol.l_policy_idx[:, li_zero, gi_low, s_bad, ei_mid]
good_bp = sol.b_policy_idx[:, li_zero, gi_low, s_good, ei_mid]
good_lp = sol.l_policy_idx[:, li_zero, gi_low, s_good, ei_mid]
bad_n = [sol.n[bad_bp[bi], bad_lp[bi], gi_low, s_bad] for bi in 1:model.Nb]
good_n = [sol.n[good_bp[bi], good_lp[bi], gi_low, s_good] for bi in 1:model.Nb]
bad_repay = .!sol.d[:, li_zero, gi_low, s_bad, ei_mid]
good_repay = .!sol.d[:, li_zero, gi_low, s_good, ei_mid]

p_nb = plot(
    model.b[bad_repay],
    bad_n[bad_repay],
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "private issuance n",
    title = "Private issuance policy n(b) (σ = 4%, Δ(g_L,s_B)=30%, Δ=0 elsewhere, l = 0)",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (1100, 650),
)
plot!(p_nb, model.b[good_repay], good_n[good_repay], color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

bad_lp_nd = sol.l_policy_idx[:, li_zero, gi_low, s_bad, ei_mid]
good_lp_nd = sol.l_policy_idx[:, li_zero, gi_low, s_good, ei_mid]
bad_lp_d = sol.l_policy_idx_d[:, li_zero, gi_low, s_bad, ei_mid]
good_lp_d = sol.l_policy_idx_d[:, li_zero, gi_low, s_good, ei_mid]
bad_nl_nd = (model.g[gi_low] .* model.l[bad_lp_nd]) ./ model.R_l
good_nl_nd = (model.g[gi_low] .* model.l[good_lp_nd]) ./ model.R_l
bad_nl_d = (model.g[gi_low] .* model.l[bad_lp_d]) ./ model.R_l
good_nl_d = (model.g[gi_low] .* model.l[good_lp_d]) ./ model.R_l

p_nl_nd = plot(
    model.b,
    bad_nl_nd,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "LOLR issuance n_l",
    title = "ND policy n_l(b) (σ = 4%, Δ(g_L,s_B)=30%, Δ=0 elsewhere, l = 0)",
    legend = :topright,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (700, 600),
)
plot!(p_nl_nd, model.b, good_nl_nd, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

p_nl_d = plot(
    model.b,
    bad_nl_d,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "LOLR issuance n_l",
    title = "D policy n_l(b) (σ = 4%, Δ(g_L,s_B)=30%, Δ=0 elsewhere, l = 0)",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (700, 600),
)
plot!(p_nl_d, model.b, good_nl_d, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")
p_nl = plot(p_nl_nd, p_nl_d, layout = (1, 2), size = (1350, 550))

d_bad = Float64.(sol.d[:, li_zero, gi_low, s_bad, ei_mid])
d_good = Float64.(sol.d[:, li_zero, gi_low, s_good, ei_mid])
p_default = plot(
    model.b,
    d_bad,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "default decision d",
    title = "Default policy d(b) (σ = 4%, Δ(g_L,s_B)=30%, Δ=0 elsewhere, l = 0)",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    ylims = (-0.05, 1.05),
    size = (1100, 650),
)
plot!(p_default, model.b, d_good, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

schedule_path = joinpath("result", "s_lolr_v2_sigma4_delta30_else0_" * "low_growth_selected_schedule.png")
nb_path = joinpath("result", "s_lolr_v2_sigma4_delta30_else0_" * "low_growth_private_policy.png")
nl_path = joinpath("result", "s_lolr_v2_sigma4_delta30_else0_" * "low_growth_lolr_policy.png")
default_path = joinpath("result", "s_lolr_v2_sigma4_delta30_else0_" * "low_growth_default_policy.png")

savefig(p_schedule, schedule_path)
savefig(p_nb, nb_path)
savefig(p_nl, nl_path)
savefig(p_default, default_path)

(
    schedule_path = schedule_path,
    private_policy_path = nb_path,
    lolr_policy_path = nl_path,
    default_path = default_path,
)


(schedule_path = "result/s_lolr_v2_sigma4_delta30_else0_low_growth_selected_schedule.png", private_policy_path = "result/s_lolr_v2_sigma4_delta30_else0_low_growth_private_policy.png", lolr_policy_path = "result/s_lolr_v2_sigma4_delta30_else0_low_growth_lolr_policy.png", default_path = "result/s_lolr_v2_sigma4_delta30_else0_low_growth_default_policy.png")

## Save Solved Scenario Bundle

In [5]:

# Save the solved scenario bundle for later comparison tables.
scenario_meta = (
    name = "σ=4, ΔLB=30, Δelse=0",
    header_top = raw"S4-30/0",
    header_bottom = raw"$\sigma=4\%,\ \Delta_{ND,LB}=30\%,\ \Delta_{else}=0\%$",
    bundle_name = "s_lolr_v2_case_sigma4_delta30_else0.jls",
)

bundle_path = joinpath("result", scenario_meta.bundle_name)
mkpath(dirname(bundle_path))
open(bundle_path, "w") do io
    serialize(io, (scenario = scenario_meta, model = model, sol = sol))
end
println(bundle_path)


result/s_lolr_v2_case_sigma4_delta30_else0.jls
